# Retrieval Evaluation

This notebook evaluates the CRMP vector retriever against a small set of manually labeled legal questions. It measures ranking quality with Hit@K, Recall@K, and Mean Reciprocal Rank, while also recording end-to-end retrieval latency.

Detailed results and an aggregate summary are saved under `data/evaluation/`.


## 1. Import the required libraries

Load timing, HTTP, numerical, tabular, and Qdrant utilities for evaluation.


In [1]:
import time
import requests
import numpy as np
import pandas as pd

from qdrant_client import QdrantClient

## 2. Configure services and metrics

Set model and collection details and define the ranking cutoffs to evaluate.


In [2]:
QDRANT_URL = "http://localhost:6333"

COLLECTION_NAME = "crmp_bge_m3"

OLLAMA_URL = "http://localhost:11434/api/embed"

EMBEDDING_MODEL = "bge-m3"

K_VALUES = [1, 3, 5, 10]

## 3. Connect to the indexed collection

Initialize Qdrant and verify the collection and point count.


In [3]:
client = QdrantClient(
    url=QDRANT_URL
)

info = client.get_collection(
    COLLECTION_NAME
)

print(f"Collection: {COLLECTION_NAME}")
print(f"Points: {info.points_count}")

Collection: crmp_bge_m3
Points: 1280


## 4. Define the evaluation queries

Create labeled questions with the legal articles considered relevant.


In [4]:
# Relevant articles are manually labeled ground truth for each query.
evaluation_queries = [
    {
        "id": "q001",
        "query": "Quais são os princípios que orientam a atuação do Município na prossecução do interesse público?",
        "relevant_articles": [
            "A-1/1.º"
        ]
    },
    {
        "id": "q002",
        "query": "Que regras existem relativamente à proteção de dados pessoais?",
        "relevant_articles": [
            "A-1/7.º"
        ]
    },
    {
        "id": "q003",
        "query": "Como deve ser apresentado um requerimento ao Município?",
        "relevant_articles": [
            "A-2/3.º"
        ]
    },
    {
        "id": "q004",
        "query": "Que elementos devem constar de um requerimento?",
        "relevant_articles": [
            "A-2/4.º"
        ]
    },
    {
        "id": "q005",
        "query": "O que acontece quando um requerimento apresenta deficiências?",
        "relevant_articles": [
            "A-2/5.º"
        ]
    }
]


## 5. Define query embedding

Create a helper that obtains `bge-m3` vectors from the local Ollama service.


In [5]:
def get_embedding(
    text,
    model=EMBEDDING_MODEL
):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    response.raise_for_status()

    return response.json()["embeddings"][0]

## 6. Define timed retrieval

Embed each question, search Qdrant, and return results with end-to-end latency.


In [6]:
def retrieve(
    query,
    limit=10
):
    query_vector = get_embedding(query)

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=limit,
        with_payload=True
    )

    return results.points

## 7. Deduplicate results by article

Convert chunk-level rankings into a unique ordered list of legal articles.


In [7]:
def get_unique_articles(results):
    # Multiple chunks may belong to one article; count each article only once.
    articles = []

    for result in results:
        article = result.payload.get("article")

        if article not in articles:
            articles.append(article)

    return articles


## 8. Inspect one evaluation example

Run the first labeled query and compare retrieved articles with the expected answer.


In [8]:
test = evaluation_queries[0]

results = retrieve(
    test["query"],
    limit=10
)

retrieved_articles = get_unique_articles(results)

print("Pergunta:")
print(test["query"])

print("\nEsperado:")
print(test["relevant_articles"])

print("\nRecuperado:")
print(retrieved_articles)

Pergunta:
Quais são os princípios que orientam a atuação do Município na prossecução do interesse público?

Esperado:
['A-1/1.º']

Recuperado:
['A-1/1.º', 'F-2/1.º', 'A-1/2.º', 'A/2.º', 'B-1/1.º', 'A-1/7.º', 'A-1/3.º', 'C-2/3.º', 'A-1/4.º', 'A-1/6.º']


## 9. Define Hit@K

Measure whether at least one relevant article appears within the first K results.


In [9]:
def hit_at_k(
    retrieved,
    relevant,
    k
):
    top_k = retrieved[:k]

    # Hit@K is binary: one relevant result is enough for a successful query.
    return int(
        any(
            article in relevant
            for article in top_k
        )
    )


## 10. Test Hit@K

Verify the metric implementation on the sample retrieval result.


In [10]:
print(
    hit_at_k(
        retrieved_articles,
        test["relevant_articles"],
        5
    )
)

1


## 11. Define Recall@K

Measure the proportion of labeled relevant articles recovered within the first K results.


In [11]:
def recall_at_k(
    retrieved,
    relevant,
    k
):
    if not relevant:
        return 0.0

    top_k = set(
        retrieved[:k]
    )

    # Set operations make overlap calculation explicit and order-independent.
    relevant = set(relevant)

    retrieved_relevant = (
        top_k & relevant
    )

    return (
        len(retrieved_relevant)
        / len(relevant)
    )


## 12. Define reciprocal rank

Score how early the first relevant article appears in the ranking.


In [12]:
def reciprocal_rank(
    retrieved,
    relevant
):
    # Reciprocal rank depends only on the first relevant result.
    for rank, article in enumerate(
        retrieved,
        start=1
    ):
        if article in relevant:
            return 1 / rank

    return 0.0


## 13. Define per-query evaluation

Calculate rankings, latency, and all configured metrics for one labeled question.


In [13]:
def evaluate_query(
    item,
    max_k=10
):
    # Measure the complete retrieval path, including query embedding.
    start = time.perf_counter()

    results = retrieve(
        item["query"],
        limit=max_k
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    retrieved_articles = (
        get_unique_articles(results)
    )

    result = {
        "id": item["id"],
        "query": item["query"],
        "relevant_articles": item["relevant_articles"],
        "retrieved_articles": retrieved_articles,
        "reciprocal_rank": reciprocal_rank(
            retrieved_articles,
            item["relevant_articles"]
        ),
        "latency_seconds": elapsed
    }

    for k in K_VALUES:
        result[f"hit@{k}"] = hit_at_k(
            retrieved_articles,
            item["relevant_articles"],
            k
        )

        result[f"recall@{k}"] = recall_at_k(
            retrieved_articles,
            item["relevant_articles"],
            k
        )

    return result


## 14. Evaluate the full query set

Run every labeled question and collect detailed results.


In [14]:
evaluation_results = []

for index, item in enumerate(
    evaluation_queries,
    start=1
):
    result = evaluate_query(
        item,
        max_k=max(K_VALUES)
    )

    evaluation_results.append(
        result
    )

    print(
        f"[{index}/{len(evaluation_queries)}] "
        f"{item['id']} "
        f"- RR={result['reciprocal_rank']:.3f}"
    )

[1/5] q001 - RR=1.000
[2/5] q002 - RR=1.000
[3/5] q003 - RR=1.000
[4/5] q004 - RR=1.000
[5/5] q005 - RR=1.000


## 15. Create the results table

Convert per-query dictionaries into a pandas DataFrame for inspection and export.


In [15]:
df_results = pd.DataFrame(
    evaluation_results
)

df_results

,id,query,relevant_articles,retrieved_articles,reciprocal_rank,latency_seconds,hit@1,recall@1,hit@3,recall@3,hit@5,recall@5,hit@10,recall@10
0,q001,Quais são os princípios que orientam a atuação...,[A-1/1.º],"[A-1/1.º, F-2/1.º, A-1/2.º, A/2.º, B-1/1.º, A-...",1.0,1.060943,1,1.0,1,1.0,1,1.0,1,1.0
1,q002,Que regras existem relativamente à proteção de...,[A-1/7.º],"[A-1/7.º, E-7/13.º, A-2/4.º, D-12/7.º, C-3/15....",1.0,0.854311,1,1.0,1,1.0,1,1.0,1,1.0
2,q003,Como deve ser apresentado um requerimento ao M...,[A-2/3.º],"[A-2/3.º, D-4/15.º-C, F-2/3.º, D-3/55.º, E-7/8...",1.0,0.870092,1,1.0,1,1.0,1,1.0,1,1.0
3,q004,Que elementos devem constar de um requerimento?,[A-2/4.º],"[A-2/4.º, D-3/55.º, B-1/5.º-A, E-7/8.º-A, 24.º...",1.0,0.822882,1,1.0,1,1.0,1,1.0,1,1.0
4,q005,O que acontece quando um requerimento apresent...,[A-2/5.º],"[A-2/5.º, C-2/22.º, F-2/5.º, D-9/7.º, A-2/6.º,...",1.0,0.938079,1,1.0,1,1.0,1,1.0,1,1.0


## 16. Aggregate evaluation metrics

Compute mean Hit@K, Recall@K, MRR, and latency across all questions.


In [16]:
metrics = {}

for k in K_VALUES:
    metrics[f"Hit@{k}"] = (
        df_results[f"hit@{k}"].mean()
    )

    metrics[f"Recall@{k}"] = (
        df_results[f"recall@{k}"].mean()
    )

metrics["MRR"] = (
    df_results["reciprocal_rank"].mean()
)

metrics["Mean Latency"] = (
    df_results["latency_seconds"].mean()
)

## 17. Display the aggregate results

Print the summary metrics in a readable format.


In [17]:
for metric, value in metrics.items():
    print(
        f"{metric}: {value:.4f}"
    )

Hit@1: 1.0000
Recall@1: 1.0000
Hit@3: 1.0000
Recall@3: 1.0000
Hit@5: 1.0000
Recall@5: 1.0000
Hit@10: 1.0000
Recall@10: 1.0000
MRR: 1.0000
Mean Latency: 0.9093


## 18. Export detailed results

Write the per-query evaluation table to CSV.


In [18]:
from pathlib import Path

# Run this notebook from notebooks/ so evaluation files resolve correctly.
PROJECT_ROOT = Path.cwd().parent

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "bge_m3_retrieval_results.csv"
)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_results.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(OUTPUT_FILE)


c:\Users\user\Documents\GitHub\legal-rag-pt\data\evaluation\bge_m3_retrieval_results.csv


## 19. Build the evaluation summary

Package model metadata, query count, and aggregate metrics.


In [19]:
summary = {
    "model": EMBEDDING_MODEL,
    "collection": COLLECTION_NAME,
    "num_queries": len(evaluation_queries),
    **{
        key: float(value)
        for key, value in metrics.items()
    }
}

## 20. Save the summary

Serialize the aggregate evaluation report as UTF-8 JSON.


In [20]:
import json

SUMMARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "bge_m3_summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,  # Keep Portuguese query text readable in the summary.
        indent=2
    )
